## Rules

Only exclude the edge if all three are true:
1. Same subject entity
2. Same object entity
3. Predicate is a paraphrase of an existing predicate
(embedding similarity ≥ 0.9)
AND
You have already recorded the surface form elsewhere

else accept the edge.

| Case                                | Action                   |
| ----------------------------------- | ------------------------ |
| Same entities, exact same predicate | Exclude (true duplicate) |
| Same entities, paraphrase predicate | Accept + normalize       |
| Same entities, broader predicate    | Accept                   |
| Same entities, different relation   | Accept                   |
| Same entities, conflicting relation | Accept + flag            |


## My current entity resolution pipeline

A. Blocking (Generate Candidates)
- FAISS based blocking

B. Entity Resolution (Check similarity)
- String similarity
- Embedding similarity 
- Neighborhood similarity

C. Merging (Merge similar entities)
- Merge entities threshold is met
 

## Improvements (in priority order)
1. Predicate canonicalization
This will:

- Improve neighbor overlap
- Reduce graph fragmentation
- Improve query quality

2. Confidence-weighted merging

Instead of deciding to MERGE or NO MERGE

Use a confidence weighted merging e.g(MERGE (0.92 confidence))
This unlocks:
- Auditing
- Rollbacks
- Batch refinement

## My pipeline currently uses:
- FAISS blocking
- String similarity
- Embedding similarity
- Neighbor overlap

It does not have:
- Entity typing (added)
- Contextual mention embeddings (added)
- Predicate semantics
- Sense disambiguation

## Upgrade road map

- Phase 1 (low effort, high signal)
    - Entity typing (even rule-based)
    - Predicate embedding clustering
    - Logging only (no splitting yet)

- Phase 2 (medium effort)
    - Contextual mention embeddings
    - Predicate-conditioned similarity

- Phase 3 (high effort, high reward)
    - Mention-level entities
    - Sense-aware merging

##  To do

- Check why neighbor similarity is not working (its working)
- Convert this into a class-based pipeline
- Add FAISS-based blocking (done)
- Add entity embedding updates after merge

## Pipeline evolution

- String similarity
- Embeddings
- FAISS blocking
- Neighborhood similarity
- Entity typing
- Contextual mentions
- Predicate semantics


| Ambiguity              | Solution      | Complexity |
| ---------------------- | ------------------ | ---------------------- |
| Typos / abbreviations  | String sim         | Cheap                  |
| Synonyms / paraphrases | Embeddings         | Costly but general     |
| Alias vs same name     | FAISS              | Indexing + state       |
| Polysemy               | Typing             | Extra model            |
| Role-dependent meaning | Context embeddings | More inference         |
| Semantic drift         | Predicate modeling | Graph semantics        |


In [1]:
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer, util

from generate_graph import get_propositions, generateEdges, createGraph, get_propositions_nosplit
from refine_graph_v4 import find_merge_candidates
from query_graph import QueryGraph
from tqdm import tqdm
from fuzzywuzzy import fuzz
from knowledge_graph_maker import Edge, Node

import pandas as pd
tqdm.pandas()

/Users/user/miniconda3/envs/graphmaker/lib/python3.11/site-packages/fuzzywuzzy/fuzz.py:11: UserWarning: Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning
  warnings.warn('Using slow pure-python SequenceMatcher. Install python-Levenshtein to remove this warning')


In [2]:
# model = SentenceTransformer("all-MiniLM-L6-v2")

model = SentenceTransformer("all-mpnet-base-v2")

# model = SentenceTransformer("E5-Base")


In [ ]:
from knowledge_graph_maker import Edge, Node

list_of_edges2 = [Edge(node_1=Node(label='Person', name='John Doe', properties={'occupation': 'software engineer', 'rank': 'employee', 'birth_place': ''}), node_2=Node(label='Organization', name='Microsoft Corporation', properties={}), relationship='works at', metadata={'summary': 'John Doe works at Microsoft Corporation as a software engineer.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=0),
 Edge(node_1=Node(label='Person', name='J. Doe', properties={'occupation': 'employee'}), node_2=Node(label='Organization', name='Microsoft Corporation', properties={}), relationship='works at', metadata={'summary': 'J. Doe is an employee of Microsoft.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=1),
 
 Edge(node_1=Node(label='Person', name='Alice Smith', properties={'occupation': 'Employee', 'rank': 'N/A', 'birth_place': 'N/A'}), node_2=Node(label='Organization', name='Google LLC', properties={}), relationship='employee of', metadata={'summary': 'Alice Smith is employed by Google LLC.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=2),
 
 Edge(node_1=Node(label='Person', name='A. Smith', properties={'occupation': 'Employee'}), node_2=Node(label='Organization', name='Google', properties={}), relationship='works at', metadata={'summary': 'A. Smith works at Google.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=3),
 
 Edge(node_1=Node(label='Organization', name='Amazon', properties={}), node_2=Node(label='Object', name='Alexa', properties={}), relationship='develops', metadata={'summary': 'Amazon develops the voice assistant Alexa.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=4),
 
 Edge(node_1=Node(label='Organization', name='Amazon Inc.', properties={}), node_2=Node(label='Object', name='Alexa voice assistant', properties={}), relationship='produces', metadata={'summary': 'Amazon Inc. produces the Alexa voice assistant.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=5),
 
 Edge(node_1=Node(label='Person', name='Elon Musk', properties={'occupation': 'CEO', 'birth_place': 'Pretoria, South Africa'}), node_2=Node(label='Organization', name='SpaceX', properties={'type': 'Aerospace manufacturer', 'founded': '2002'}), relationship='leads', metadata={'summary': 'Elon Musk leads SpaceX.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=6),
 
 Edge(node_1=Node(label='Person', name='Elon Musk', properties={'occupation': 'CEO', 'birth_place': '', 'rank': ''}), node_2=Node(label='Organization', name='Space Exploration Technologies', properties={}), relationship='CEO of', metadata={'summary': 'Elon Musk is the CEO of Space Exploration Technologies.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=7),
 
 Edge(node_1=Node(label='Organization', name='Apple', properties={}), node_2=Node(label='Person', name='Steve Jobs', properties={'occupation': 'Entrepreneur', 'rank': 'Founder'}), relationship='founded by', metadata={'summary': 'Apple was founded by Steve Jobs.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=8),
 
 Edge(node_1=Node(label='Organization', name='Apple Inc.', properties={}), node_2=Node(label='Person', name='S. Jobs', properties={}), relationship='co-founded by', metadata={'summary': 'Apple Inc. was co-founded by S. Jobs.', 'generated_at': '2025-10-30 17:34:07.282620'}, order=9)]


toy_data = []
for edge in list_of_edges2:
# unique_entities.add(edge.node_1.name)
# unique_entities.add(edge.node_2.name)

    toy_data.append({
        "subject": edge.node_1.name,
        "predicate": edge.relationship,
        "object": edge.node_2.name     
        # "context": edge.metadata.get("summary"),
    })

## V1

Components

- Semantic blocking
- String similarity
- Neighbor overlap

In [ ]:
# ----------------------------
# Toy Dataset
# ----------------------------
toy_dataset = [
    {"subject": "John Doe", "predicate": "works at", "object": "Microsoft"},
    {"subject": "J. Doe", "predicate": "employee of", "object": "Microsoft"},
    {"subject": "Alice Smith", "predicate": "works at", "object": "Google"},
    {"subject": "A. Smith", "predicate": "works at", "object": "Google"},
    {"subject": "Amazon", "predicate": "develops", "object": "Alexa"},
    {"subject": "Amazon Inc.", "predicate": "produces", "object": "Alexa voice assistant"},
    {"subject": "Elon Musk", "predicate": "leads", "object": "SpaceX"},
    {"subject": "Elon Musk", "predicate": "CEO of", "object": "Space Exploration Technologies"},
    {"subject": "Apple", "predicate": "founded by", "object": "Steve Jobs"},
    {"subject": "Apple Inc.", "predicate": "co-founded by", "object": "S. Jobs"},
]

# ----------------------------
# 1: Helper functions
# ----------------------------

def string_sim(a, b):
    """Normalized string similarity between 0 and 1"""
    return fuzz.token_sort_ratio(a, b) / 100

def extract_entities(triple):
    return [
        {"text": triple["subject"], "role": "subject"},
        {"text": triple["object"], "role": "object"},
    ]

# ----------------------------
# 2: Initialize entity store
# ----------------------------

entity_store = {}
entity_ids = []
entity_id_counter = 1

existing_edge_keys = set()  # to track duplicates
accepted_edges = []
excluded_edges = []

# ----------------------------
# 3: Semantic blocking (toy: all existing entities)
# ----------------------------
def semantic_blocking(entity_text, entity_store, threshold=0.7):
    """Return candidate entity IDs if similarity > threshold"""
    candidates = []
    for eid, info in entity_store.items():
        score = string_sim(entity_text, info["name"])
        if score >= threshold:
            print("candidate found:", eid, info["name"], "score:", score)
            candidates.append({"entity_id": eid, "score": score})
    return candidates

# ----------------------------
# 4: Pre-insertion filtering
# ----------------------------
def pre_insertion_filtering(mention_text, candidates, threshold=0.75):
    """Return the best candidate if score >= threshold, else None"""
    best_score = 0
    best_entity_id = None
    for c in candidates:
        if c["score"] > best_score:
            best_score = c["score"]
            best_entity_id = c["entity_id"]
    if best_score >= threshold:
        return best_entity_id
    return None

def neighbor_overlap(new_neighbors, existing_neighbors):
    if not new_neighbors or not existing_neighbors:
        return 0.0
    intersection = new_neighbors.intersection(existing_neighbors)
    union = new_neighbors.union(existing_neighbors)
    return len(intersection) / len(union)

# ----------------------------
# 5: Resolve entity (MERGE or INSERT)
# ----------------------------
def resolve_entity(entity_text):
    global entity_id_counter
    # blocking
    candidates = semantic_blocking(entity_text, entity_store)
    # filtering
    resolved_id = pre_insertion_filtering(entity_text, candidates)
    if resolved_id is not None:
        return resolved_id, "MERGE"
    # INSERT new entity
    new_id = f"E{entity_id_counter}"
    entity_id_counter += 1
    entity_store[new_id] = {"name": entity_text, "neighbors": set()}
    entity_ids.append(new_id)
    return new_id, "INSERT"

# ----------------------------
# 6: Insert edge & update neighbors
# ----------------------------
def insert_edge(subject_id, predicate, object_id):
    # neighbors: ego graph
    entity_store[subject_id]["neighbors"].add(f"{predicate}:{object_id}")
    entity_store[object_id]["neighbors"].add(f"{predicate}:{subject_id}")

# ----------------------------
# 7: Run the pipeline
# ----------------------------
for triple in toy_dataset:
    mentions = extract_entities(triple)
    resolved = {}
    actions = {}

    print("========================")
    # resolve each entity
    for ent in mentions:
        print("Resolving entity:", ent["text"])
        eid, action = resolve_entity(ent["text"])
        resolved[ent["role"]] = eid
        actions[ent["role"]] = action

    # create edge key to check duplicates
    edge_key = (resolved["subject"], triple["predicate"], resolved["object"])
    print("existing_edge_keys:", existing_edge_keys)
    print("edge_key:", edge_key)
    if edge_key in existing_edge_keys:
        print("dup")
        excluded_edges.append({
            **triple,
            "reason": "duplicate_edge",
            "subject_id": resolved["subject"],
            "object_id": resolved["object"]
        })
    else:
        accepted_edges.append({
            **triple,
            "subject_id": resolved["subject"],
            "object_id": resolved["object"],
            "subject_action": actions["subject"],
            "object_action": actions["object"]
        })
        existing_edge_keys.add(edge_key)
        # update neighbors
        insert_edge(resolved["subject"], triple["predicate"], resolved["object"])


# ----------------------------
# Print results
# ----------------------------
print("=== Accepted Edges ===")
for e in accepted_edges:
    print(e)

print("\n=== Excluded Edges ===")
for e in excluded_edges:
    print(e)

print("\n=== Entity Store ===")
for eid, info in entity_store.items():
    print(eid, info)


## V2

Components

- Semantic blocking
- String similarity
- Neighbor overlap

In [ ]:
# ----------------------------
# Toy Dataset
# ----------------------------
toy_dataset = [
    {"subject": "John Doe", "predicate": "works at", "object": "Microsoft"},
    {"subject": "J. Doe", "predicate": "works at", "object": "Microsoft"},
    {"subject": "Alice Smith", "predicate": "works at", "object": "Google"},
    {"subject": "A. Smith", "predicate": "works at", "object": "Google"},
    {"subject": "Amazon", "predicate": "develops", "object": "Alexa"},
    {"subject": "Amazon Inc.", "predicate": "produces", "object": "Alexa voice assistant"},
    {"subject": "Elon Musk", "predicate": "leads", "object": "SpaceX"},
    {"subject": "Elon Musk", "predicate": "CEO of", "object": "Space Exploration Technologies"},
    {"subject": "Apple", "predicate": "founded by", "object": "Steve Jobs"},
    {"subject": "Apple Inc.", "predicate": "co-founded by", "object": "S. Jobs"},
]

# ----------------------------
# Helper functions
# ----------------------------

def string_sim(a, b):
    """Normalized string similarity between 0 and 1"""
    return fuzz.token_sort_ratio(a, b) / 100

def extract_entities(triple):
    return [
        {"text": triple["subject"], "role": "subject"},
        {"text": triple["object"], "role": "object"},
    ]

# ----------------------------
# Initialize entity store
# ----------------------------

entity_store = {}
entity_ids = []
entity_id_counter = 1

existing_edge_keys = set()  # to track duplicates
accepted_edges = []
excluded_edges = []

# ----------------------------
# Semantic blocking (toy: all existing entities)
# ----------------------------
def semantic_blocking(entity_text, entity_store, threshold=0.7):
    """Return candidate entity IDs if similarity > threshold"""
    candidates = []
    for eid, info in entity_store.items():
        score = string_sim(entity_text, info["name"])
        if score >= threshold:
            print("semantic_blocking")
            print("candidate found:", eid, info["name"], "score:", score)
            candidates.append({"entity_id": eid, "score": score})
    return candidates


def neighbor_overlap(new_neighbors, existing_neighbors):

    print("new_neighbors:", new_neighbors)
    print("existing_neighbors:", existing_neighbors)

    if not new_neighbors or not existing_neighbors:
        return 0.0
    intersection = new_neighbors.intersection(existing_neighbors)
    union = new_neighbors.union(existing_neighbors)

    return len(intersection) / len(union)

# ----------------------------
# Resolve entity (MERGE or INSERT)
# ----------------------------

def resolve_entity(entity_text, role, predicate, other_entity_id=None):
    """
    role: 'subject' or 'object'
    predicate: predicate of the current triple
    other_entity_id: resolved ID of the other entity in the triple (if exists)
    """
    global entity_id_counter

    # Blocking
    candidates = semantic_blocking(entity_text, entity_store)

    best_candidate = None
    best_score = 0

    for c in candidates:
        existing = entity_store[c["entity_id"]]

        # String similarity
        s_sim = string_sim(entity_text, existing["name"])

        # Neighbor similarity (if possible)
        new_neighbors = set()
        if other_entity_id is not None:
            new_neighbors.add(f"{predicate}:{other_entity_id}")

        n_sim = neighbor_overlap(new_neighbors, existing["neighbors"])

        print(f"Evaluating candidate {c['entity_id']}: s_sim={s_sim}, n_sim={n_sim}")
        # Merge decision (gated)
        if (s_sim >= 0.9 or (s_sim >= 0.75 and n_sim >= 0.5)):
            if s_sim > best_score:
                best_candidate = c["entity_id"]
                best_score = s_sim

    if best_candidate is not None:
        return best_candidate, "MERGE"

    # INSERT
    new_id = f"E{entity_id_counter}"
    entity_id_counter += 1
    entity_store[new_id] = {
        "name": entity_text,
        "neighbors": set()
    }
    return new_id, "INSERT"

# ----------------------------
# Insert edge & update neighbors
# ----------------------------
def insert_edge(subject_id, predicate, object_id):
    # neighbors: ego graph
    entity_store[subject_id]["neighbors"].add(f"{predicate}:{object_id}")
    entity_store[object_id]["neighbors"].add(f"{predicate}:{subject_id}")

# ----------------------------
# Run the pipeline
# ----------------------------
for triple in toy_dataset:
    subject_text = triple["subject"]
    object_text = triple["object"]
    predicate = triple["predicate"]

    print("========================")
    print("Resolving triple:", triple)
    # resolve subject first
    subj_id, subj_action = resolve_entity(
        subject_text,
        role="subject",
        predicate=predicate
    )

    # resolve object (can use subject as neighbor)
    obj_id, obj_action = resolve_entity(
        object_text,
        role="object",
        predicate=predicate,
        other_entity_id=subj_id
    )

    # canonical_pred = PREDICATE_MAP.get(predicate, predicate)
    edge_key = (subj_id, predicate, obj_id)

    if edge_key in existing_edge_keys:
        excluded_edges.append({
            **triple,
            "reason": "duplicate_edge",
            "subject_id": subj_id,
            "object_id": obj_id
        })
        continue

    accepted_edges.append({
        **triple,
        "subject_id": subj_id,
        "object_id": obj_id,
        "subject_action": subj_action,
        "object_action": obj_action
    })

    existing_edge_keys.add(edge_key)

    # update neighbors (MERGE semantics)
    entity_store[subj_id]["neighbors"].add(f"{predicate}:{obj_id}")
    entity_store[obj_id]["neighbors"].add(f"{predicate}:{subj_id}")

# ----------------------------
# Print results
# ----------------------------
print("=== Accepted Edges ===")
for e in accepted_edges:
    print(e)

print("\n=== Excluded Edges ===")
for e in excluded_edges:
    print(e)

print("\n=== Entity Store ===")
for eid, info in entity_store.items():
    print(eid, info)


## V3

Components

- Semantic blocking
- String similarity
- Embedding similarity
- Neighbor overlap

In [ ]:
# ----------------------------
# Toy Dataset
# ----------------------------
toy_dataset = [
    {"subject": "John Doe", "predicate": "works at", "object": "Microsoft"},
    {"subject": "J. Doe", "predicate": "works at", "object": "Microsoft"},
    {"subject": "Alice Smith", "predicate": "works at", "object": "Google"},
    {"subject": "A. Smith", "predicate": "works at", "object": "Google"},
    {"subject": "Amazon", "predicate": "develops", "object": "Alexa"},
    {"subject": "Amazon Inc.", "predicate": "produces", "object": "Alexa voice assistant"},
    {"subject": "Elon Musk", "predicate": "leads", "object": "SpaceX"},
    {"subject": "Elon Musk", "predicate": "CEO of", "object": "Space Exploration Technologies"},
    {"subject": "Apple", "predicate": "founded by", "object": "Steve Jobs"},
    {"subject": "Apple Inc.", "predicate": "co-founded by", "object": "S. Jobs"},
]

# ----------------------------
# Helper functions
# ----------------------------
def string_sim(a, b):
    """Normalized string similarity between 0 and 1"""
    return fuzz.token_sort_ratio(a, b) / 100

def embedding_sim(a, b):
    """Cosine similarity between two strings"""
    emb_a = model.encode(a, convert_to_tensor=True)
    emb_b = model.encode(b, convert_to_tensor=True)
    return util.cos_sim(emb_a, emb_b).item()

def extract_entities(triple):
    return [
        {"text": triple["subject"], "role": "subject"},
        {"text": triple["object"], "role": "object"},
    ]

# ----------------------------
# Initialize entity store
# ----------------------------

entity_store = {}
entity_ids = []
entity_id_counter = 1

existing_edge_keys = set()  # to track duplicates
accepted_edges = []
excluded_edges = []

# ----------------------------
# Semantic blocking (toy: all existing entities)
# ----------------------------

def semantic_blocking(entity_text, entity_store, threshold=0.6):
    """
    Hybrid blocking using string + embedding similarity
    """
    candidates = []
    query_emb = model.encode(entity_text, convert_to_tensor=True)

    for eid, info in entity_store.items():
        s_sim = string_sim(entity_text, info["name"])
        e_sim = util.cos_sim(query_emb, info["embedding"]).item()

        # loose blocking, strict merge later
        if max(s_sim, e_sim) >= threshold:
            print("semantic_blocking")
            print(
                f"candidate: {eid}, name={info['name']}, "
                f"s_sim={s_sim:.3f}, e_sim={e_sim:.3f}"
            )
            candidates.append({
                "entity_id": eid,
                "s_sim": s_sim,
                "e_sim": e_sim
            })
    return candidates



def neighbor_overlap(new_neighbors, existing_neighbors):

    print("new_neighbors:", new_neighbors)
    print("existing_neighbors:", existing_neighbors)

    if not new_neighbors or not existing_neighbors:
        return 0.0
    intersection = new_neighbors.intersection(existing_neighbors)
    union = new_neighbors.union(existing_neighbors)

    return len(intersection) / len(union)

# ----------------------------
# Resolve entity (MERGE or INSERT)
# ----------------------------

def resolve_entity(entity_text, role, predicate, other_entity_id=None):
    global entity_id_counter

    candidates = semantic_blocking(entity_text, entity_store)

    best_candidate = None
    best_score = 0

    for c in candidates:
        existing = entity_store[c["entity_id"]]

        s_sim = c["s_sim"]
        e_sim = c["e_sim"]

        # Neighbor similarity
        new_neighbors = set()
        if other_entity_id is not None:
            new_neighbors.add(f"{predicate}:{other_entity_id}")

        n_sim = neighbor_overlap(new_neighbors, existing["neighbors"])

        # Weighted fusion score
        fused_score = 0.4 * s_sim + 0.4 * e_sim + 0.2 * n_sim

        print(
            f"Evaluating {c['entity_id']}: "
            f"s_sim={s_sim:.3f}, e_sim={e_sim:.3f}, "
            f"n_sim={n_sim:.3f}, fused={fused_score:.3f}"
        )

        # Merge rules (interpretable + safe)
        if (e_sim >= 0.85 or (s_sim >= 0.8 and e_sim >= 0.7) or (fused_score >= 0.75)
        ):
            if fused_score > best_score:
                best_candidate = c["entity_id"]
                best_score = fused_score

    if best_candidate is not None:
        return best_candidate, "MERGE"

    # INSERT
    new_id = f"E{entity_id_counter}"
    entity_id_counter += 1

    entity_store[new_id] = {
        "name": entity_text,
        "embedding": model.encode(entity_text, convert_to_tensor=True),
        "neighbors": set()
    }

    return new_id, "INSERT"

# ----------------------------
# Insert edge & update neighbors
# ----------------------------
def insert_edge(subject_id, predicate, object_id):
    # neighbors: ego graph
    entity_store[subject_id]["neighbors"].add(f"{predicate}:{object_id}")
    entity_store[object_id]["neighbors"].add(f"{predicate}:{subject_id}")

# ----------------------------
# Run the pipeline
# ----------------------------
for triple in toy_dataset:
    subject_text = triple["subject"]
    object_text = triple["object"]
    predicate = triple["predicate"]

    print("========================")
    print("Resolving triple:", triple)
    # resolve subject first
    subj_id, subj_action = resolve_entity(
        subject_text,
        role="subject",
        predicate=predicate
    )

    # resolve object (can use subject as neighbor)
    obj_id, obj_action = resolve_entity(
        object_text,
        role="object",
        predicate=predicate,
        other_entity_id=subj_id
    )

    # canonical_pred = PREDICATE_MAP.get(predicate, predicate)
    edge_key = (subj_id, predicate, obj_id)

    if edge_key in existing_edge_keys:
        excluded_edges.append({
            **triple,
            "reason": "duplicate_edge",
            "subject_id": subj_id,
            "object_id": obj_id
        })
        continue

    accepted_edges.append({
        **triple,
        "subject_id": subj_id,
        "object_id": obj_id,
        "subject_action": subj_action,
        "object_action": obj_action
    })

    existing_edge_keys.add(edge_key)

    # update neighbors (MERGE semantics)
    entity_store[subj_id]["neighbors"].add(f"{predicate}:{obj_id}")
    entity_store[obj_id]["neighbors"].add(f"{predicate}:{subj_id}")

# ----------------------------
# Print results
# ----------------------------
print("=== Accepted Edges ===")
for e in accepted_edges:
    print(e)

print("\n=== Excluded Edges ===")
for e in excluded_edges:
    print(e)

# print("\n=== Entity Store ===")
# for eid, info in entity_store.items():
#     print(eid, info)


## V4

Components

- FAISS embedding
- Semantic blocking
- String similarity
- Embedding similarity
- Neighbor overlap

In [ ]:
EMBED_DIM = 768  # MiniLM; use 768 for mpnet

faiss_index = faiss.IndexFlatIP(EMBED_DIM)
faiss_id_map = []  # maps FAISS row → entity_id

# ----------------------------
# Helper functions
# ----------------------------

def normalize(v):
    return v / np.linalg.norm(v)


def string_sim(a, b):
    """Normalized string similarity between 0 and 1"""
    return fuzz.token_sort_ratio(a, b) / 100

def embedding_sim(a, b):
    """Cosine similarity between two strings"""
    emb_a = model.encode(a, convert_to_tensor=True)
    emb_b = model.encode(b, convert_to_tensor=True)
    return util.cos_sim(emb_a, emb_b).item()

def extract_entities(triple):
    return [
        {"text": triple["subject"], "role": "subject"},
        {"text": triple["object"], "role": "object"},
    ]

# ----------------------------
# Initialize entity store
# ----------------------------

entity_store = {}
entity_ids = []
entity_id_counter = 1

existing_edge_keys = set()  # to track duplicates
accepted_edges = []
excluded_edges = []

# ----------------------------
# Semantic blocking (toy: all existing entities)
# ----------------------------

# def semantic_blocking(entity_text, entity_store, threshold=0.6):
#     """
#     Hybrid blocking using string + embedding similarity
#     """
#     candidates = []
#     query_emb = model.encode(entity_text, convert_to_tensor=True)

#     for eid, info in entity_store.items():
#         s_sim = string_sim(entity_text, info["name"])
#         e_sim = util.cos_sim(query_emb, info["embedding"]).item()

#         # loose blocking, strict merge later
#         if max(s_sim, e_sim) >= threshold:
#             print("semantic_blocking")
#             print(
#                 f"candidate: {eid}, name={info['name']}, "
#                 f"s_sim={s_sim:.3f}, e_sim={e_sim:.3f}"
#             )
#             candidates.append({
#                 "entity_id": eid,
#                 "s_sim": s_sim,
#                 "e_sim": e_sim
#             })
#     return candidates

def semantic_blocking_faiss(entity_text, top_k=10, sim_threshold=0.7):
    if faiss_index.ntotal == 0:
        return []

    query_emb = model.encode(entity_text)
    query_emb = normalize(query_emb).reshape(1, -1)

    scores, indices = faiss_index.search(query_emb, top_k)

    candidates = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue

        if score >= sim_threshold:
            eid = faiss_id_map[idx]
            candidates.append({
                "entity_id": eid,
                "e_sim": float(score)
            })

    return candidates


def neighbor_overlap(new_neighbors, existing_neighbors):

    print("new_neighbors:", new_neighbors)
    print("existing_neighbors:", existing_neighbors)

    if not new_neighbors or not existing_neighbors:
        return 0.0
    intersection = new_neighbors.intersection(existing_neighbors)
    union = new_neighbors.union(existing_neighbors)

    return len(intersection) / len(union)

# ----------------------------
# Resolve entity (MERGE or INSERT)
# ----------------------------

def resolve_entity(entity_text, role, predicate, other_entity_id=None):
    global entity_id_counter

    candidates = semantic_blocking_faiss(entity_text)

    best_candidate = None
    best_score = 0

    for c in candidates:
        existing = entity_store[c["entity_id"]]

        s_sim = string_sim(entity_text, existing["name"])
        e_sim = c["e_sim"]

        new_neighbors = set()
        if other_entity_id:
            new_neighbors.add(f"{predicate}:{other_entity_id}")

        n_sim = neighbor_overlap(new_neighbors, existing["neighbors"])

        fused_score = 0.4 * s_sim + 0.4 * e_sim + 0.2 * n_sim

        print(
            f"Evaluating {c['entity_id']}: "
            f"s_sim={s_sim:.3f}, e_sim={e_sim:.3f}, "
            f"n_sim={n_sim:.3f}, fused={fused_score:.3f}"
        )

        # Merge rules (interpretable + safe)
        if (e_sim >= 0.85 or (s_sim >= 0.8 and e_sim >= 0.7) or (fused_score >= 0.75)
        ):
            if fused_score > best_score:
                best_candidate = c["entity_id"]
                best_score = fused_score

    if best_candidate is not None:
        return best_candidate, "MERGE"

    # INSERT
    new_id = f"E{entity_id_counter}"
    entity_id_counter += 1

    embedding = model.encode(entity_text)
    embedding = normalize(embedding)

    faiss_index.add(embedding.reshape(1, -1))
    faiss_id_map.append(new_id)

    entity_store[new_id] = {
        "name": entity_text,
        "embedding": embedding,
        "neighbors": set()
    }

    return new_id, "INSERT"

# ----------------------------
# Insert edge & update neighbors
# ----------------------------
def insert_edge(subject_id, predicate, object_id):
    # neighbors: ego graph
    entity_store[subject_id]["neighbors"].add(f"{predicate}:{object_id}")
    entity_store[object_id]["neighbors"].add(f"{predicate}:{subject_id}")


### Toy dataset

In [ ]:
# ----------------------------
# Toy Dataset
# ----------------------------
toy_dataset = [
    # {"subject": "John Doe", "predicate": "works at", "object": "Microsoft"},
    # {"subject": "J. Doe", "predicate": "works at", "object": "Microsoft"},
    # {"subject": "Alice Smith", "predicate": "works at", "object": "Google"},
    # {"subject": "A. Smith", "predicate": "works at", "object": "Google"},
    # {"subject": "Amazon", "predicate": "develops", "object": "Alexa"},
    # {"subject": "Amazon Inc.", "predicate": "produces", "object": "Alexa voice assistant"},
    {"subject": "Elon Musk", "predicate": "leads", "object": "SpaceX"},
    {"subject": "Elon Musk", "predicate": "CEO of", "object": "Space Exploration Technologies"},
    {"subject": "E. Musk", "predicate": "leads", "object": "SpaceX"},
    {"subject": "E. Musk", "predicate": "founder of", "object": "SpaceX"},
    # {"subject": "Apple", "predicate": "founded by", "object": "Steve Jobs"},
    # {"subject": "Apple Inc.", "predicate": "co-founded by", "object": "S. Jobs"},
]

# ----------------------------
# Run the pipeline
# ----------------------------
for triple in toy_dataset:
    subject_text = triple["subject"]
    object_text = triple["object"]
    predicate = triple["predicate"]

    print("========================")
    print("Resolving triple:", triple)
    # resolve subject first
    subj_id, subj_action = resolve_entity(
        subject_text,
        role="subject",
        predicate=predicate
    )

    # resolve object (can use subject as neighbor)
    obj_id, obj_action = resolve_entity(
        object_text,
        role="object",
        predicate=predicate,
        other_entity_id=subj_id
    )

    # canonical_pred = PREDICATE_MAP.get(predicate, predicate)
    edge_key = (subj_id, predicate, obj_id)

    if edge_key in existing_edge_keys:
        excluded_edges.append({
            **triple,
            "reason": "duplicate_edge",
            "subject_id": subj_id,
            "object_id": obj_id
        })
        continue

    accepted_edges.append({
        **triple,
        "subject_id": subj_id,
        "object_id": obj_id,
        "subject_action": subj_action,
        "object_action": obj_action
    })

    existing_edge_keys.add(edge_key)

    # update neighbors (MERGE semantics)
    entity_store[subj_id]["neighbors"].add(f"{predicate}:{obj_id}")
    entity_store[obj_id]["neighbors"].add(f"{predicate}:{subj_id}")

# ----------------------------
# Print results
# ----------------------------
print("=== Accepted Edges ===")
for e in accepted_edges:
    print(e)

print("\n=== Excluded Edges ===")
for e in excluded_edges:
    print(e)

print("\n=== Entity Store ===")
for eid, info in entity_store.items():
    print(eid, info['name'], info['neighbors'])

## V5

Components

- FAISS embedding
- Semantic blocking
- String similarity
- Contextual Embedding similarity (rules based)
- Neighbor overlap
- Entity typing

In [ ]:
EMBED_DIM = 768  # MiniLM; use 768 for mpnet

faiss_index = faiss.IndexFlatIP(EMBED_DIM)
faiss_id_map = []  # maps FAISS row → entity_id

# ----------------------------
# Helper functions
# ----------------------------

def normalize(v):
    return v / np.linalg.norm(v)


def string_sim(a, b):
    """Normalized string similarity between 0 and 1"""
    return fuzz.token_sort_ratio(a, b) / 100

# ----------------------------
# Initialize entity store
# ----------------------------

entity_store = {}
entity_ids = []
entity_id_counter = 1

existing_edge_keys = set()  # to track duplicates
accepted_edges = []
excluded_edges = []

def semantic_blocking(entity_text, top_k=10, sim_threshold=0.7):
    if faiss_index.ntotal == 0:
        return []

    query_emb = model.encode(entity_text)
    query_emb = normalize(query_emb).reshape(1, -1)

    scores, indices = faiss_index.search(query_emb, top_k)

    candidates = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue

        if score >= sim_threshold:
            eid = faiss_id_map[idx]
            candidates.append({
                "entity_id": eid,
                "e_sim": float(score)
            })

    return candidates


def assign_type(entity_text, predicate=None):
    """
    Simple rule-based type assignment based on keywords and predicates.
    """
    entity_text_lower = entity_text.lower()
    
    # Simple keyword-based type assignment
    if entity_text_lower in ["apple", "banana", "pear"]:
        return "FOOD"
    if entity_text_lower in ["apple inc.", "microsoft", "google", "amazon", "amazon inc."]:
        return "ORGANIZATION"
    if entity_text_lower in ["elon musk", "alice smith", "john doe", "j. doe", "a. smith", "steve jobs", "s. jobs"]:
        return "PERSON"
    
    # Predicate-based override
    if predicate:
        if predicate in ["released", "founded by", "works at", "ceo of", "co-founded by", "develops", "produces", "leads"]:
            return "ORGANIZATION"
        if predicate in ["harvested in", "is a type of", "grows", "eaten"]:
            return "FOOD"
    
    return "UNKNOWN"


def types_compatible(type1, type2):
    if type1 == "UNKNOWN" or type2 == "UNKNOWN":
        return True  # fallback
    return type1 == type2


def resolve_entity(entity_text, role, predicate, other_entity_id=None):
    global entity_id_counter

    # Blocking
    candidates = semantic_blocking(entity_text)

    best_candidate = None
    best_score = 0
    new_type = assign_type(entity_text, predicate)

    for c in candidates:
        existing = entity_store[c["entity_id"]]
        existing_type = existing["type"]

        # Only consider candidates with compatible types
        if not types_compatible(new_type, existing_type):
            continue

        # String similarity
        s_sim = string_sim(entity_text, existing["name"])

        # Neighbor similarity
        new_neighbors = set()
        if other_entity_id is not None:
            new_neighbors.add(f"{predicate}:{other_entity_id}")
        n_sim = neighbor_overlap(new_neighbors, existing["neighbors"])

        # Merge decision (gated)
        if (s_sim >= 0.9 or (s_sim >= 0.75 and n_sim >= 0.5)):
            if s_sim > best_score:
                best_candidate = c["entity_id"]
                best_score = s_sim

    if best_candidate is not None:
        return best_candidate, "MERGE"

    # INSERT new entity with type
    new_id = f"E{entity_id_counter}"
    entity_id_counter += 1
    entity_store[new_id] = {
        "name": entity_text,
        "type": new_type,
        "neighbors": set()
    }
    return new_id, "INSERT"

def insert_edge(subject_id, predicate, object_id):
    entity_store[subject_id]["neighbors"].add(f"{predicate}:{object_id}")
    entity_store[object_id]["neighbors"].add(f"{predicate}:{subject_id}")


In [ ]:
# ----------------------------
# Toy Dataset
# ----------------------------
toy_dataset = [
    # {"subject": "John Doe", "predicate": "works at", "object": "Microsoft"},
    # {"subject": "J. Doe", "predicate": "works at", "object": "Microsoft"},
    # {"subject": "Alice Smith", "predicate": "works at", "object": "Google"},
    # {"subject": "A. Smith", "predicate": "works at", "object": "Google"},
    # {"subject": "Amazon", "predicate": "develops", "object": "Alexa"},
    # {"subject": "Amazon Inc.", "predicate": "produces", "object": "Alexa voice assistant"},
    {"subject": "Elon Musk", "predicate": "leads", "object": "SpaceX"},
    {"subject": "Elon Musk", "predicate": "CEO of", "object": "Space Exploration Technologies"},
    {"subject": "E. Musk", "predicate": "leads", "object": "SpaceX"},
    {"subject": "E. Musk", "predicate": "founder of", "object": "SpaceX"},
    # {"subject": "Apple", "predicate": "founded by", "object": "Steve Jobs"},
    # {"subject": "Apple Inc.", "predicate": "co-founded by", "object": "S. Jobs"},
]

# ----------------------------
# Run the pipeline
# ----------------------------
for triple in toy_dataset:
    subject_text = triple["subject"]
    object_text = triple["object"]
    predicate = triple["predicate"]

    print("========================")
    print("Resolving triple:", triple)
    # resolve subject first
    subj_id, subj_action = resolve_entity(
        subject_text,
        role="subject",
        predicate=predicate
    )

    # resolve object (can use subject as neighbor)
    obj_id, obj_action = resolve_entity(
        object_text,
        role="object",
        predicate=predicate,
        other_entity_id=subj_id
    )

    # canonical_pred = PREDICATE_MAP.get(predicate, predicate)
    edge_key = (subj_id, predicate, obj_id)

    if edge_key in existing_edge_keys:
        excluded_edges.append({
            **triple,
            "reason": "duplicate_edge",
            "subject_id": subj_id,
            "object_id": obj_id
        })
        continue

    accepted_edges.append({
        **triple,
        "subject_id": subj_id,
        "object_id": obj_id,
        "subject_action": subj_action,
        "object_action": obj_action
    })

    existing_edge_keys.add(edge_key)

    # update neighbors (MERGE semantics)
    entity_store[subj_id]["neighbors"].add(f"{predicate}:{obj_id}")
    entity_store[obj_id]["neighbors"].add(f"{predicate}:{subj_id}")

# ----------------------------
# Print results
# ----------------------------
print("=== Accepted Edges ===")
for e in accepted_edges:
    print(e)

print("\n=== Excluded Edges ===")
for e in excluded_edges:
    print(e)

print("\n=== Entity Store ===")
for eid, info in entity_store.items():
    print(eid, info['name'], info['neighbors'])

## V6

Components

- FAISS embedding
- Semantic blocking
- String similarity
- Contextual Embedding similarity (using pre-trained model)
- Neighbor overlap
- Entity typing

In [3]:
from transformers import pipeline

ner_pipeline = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    aggregation_strategy="simple"
)

Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use mps:0


In [146]:
EMBED_DIM = 768  # MiniLM; use 768 for mpnet

faiss_index = faiss.IndexFlatIP(EMBED_DIM)
faiss_id_map = []  # maps FAISS row → entity_id

# ----------------------------
# Helper functions
# ----------------------------

def normalize(v):
    """
    L2 normalize a vector
    """
    return v / np.linalg.norm(v)


def string_sim(a, b):
    """Normalized string similarity between 0 and 1"""
    return fuzz.token_sort_ratio(a, b) / 100

# ----------------------------
# Initialize entity store
# ----------------------------

entity_store = {}
entity_ids = []
entity_id_counter = 1

existing_edge_keys = set()  # to track duplicates
accepted_edges = []
excluded_edges = []


# ----------------------------
# Core functions
# ----------------------------

def semantic_blocking(entity_text, top_k=10, sim_threshold=0.7):
    """
    Semantic blocking using FAISS for efficient retrieval of candidate entities based on embedding similarity.
    """
    
    if faiss_index.ntotal == 0:
        return []

    query_emb = model.encode(entity_text)
    query_emb = normalize(query_emb).reshape(1, -1)

    scores, indices = faiss_index.search(query_emb, top_k)

    candidates = []
    for score, idx in zip(scores[0], indices[0]):
        if idx == -1:
            continue

        if score >= sim_threshold:
            eid = faiss_id_map[idx]
            candidates.append({
                "entity_id": eid,
                "e_sim": float(score)
            })

    return candidates


def predict_entity_type(entity_text, predicate=None):
    """
    Predict coarse entity type using a pretrained NER model.
    Returns: PERSON | ORGANIZATION | LOCATION | MISC | UNKNOWN
    """
    # Add minimal context (important!)
    if predicate:
        text = f"{entity_text} {predicate}"
    else:
        text = entity_text

    try:
        entities = ner_pipeline(text)
    except Exception:
        return "UNKNOWN"

    for ent in entities:
        if ent["word"].lower() in entity_text.lower():
            label = ent["entity_group"]
            print(f"NER prediction for '{entity_text}': {label}")
            if label == "PER":
                return "PERSON"
            if label == "ORG":
                return "ORGANIZATION"
            if label == "LOC":
                return "LOCATION"
            if label == "MISC":
                return "MISC"

    return "UNKNOWN"

def types_compatible(type1, type2):
    """
    Simple type compatibility check.
    If either type is UNKNOWN, we consider them compatible (fallback). Otherwise, they must match exactly.
    """
    if type1 == "UNKNOWN" or type2 == "UNKNOWN":
        return True  # fallback
    return type1 == type2


def composite_similarity(entity_text, existing, predicate=None, other_entity_id=None,
                         alpha=0.4, beta=0.3, gamma=0.2, delta=0.1):
    """
    Composite similarity for entity resolution.
    Weights sum to 1: alpha + beta + gamma + delta = 1
    """
    # String similarity
    s_sim = string_sim(entity_text, existing["name"])

    # Embedding similarity (contextual)
    # Use the triple text as context for embeddings
    context_text = f"{entity_text} {predicate}" if predicate else entity_text
    emb_entity = model.encode([context_text])
    emb_existing = model.encode([existing["name"]])
    e_sim = util.cos_sim(emb_entity, emb_existing).item()  # cosine similarity

    # Neighbor similarity
    new_neighbors = set()
    if other_entity_id is not None and predicate is not None:
        new_neighbors.add(f"{predicate}:{other_entity_id}")
    n_sim = neighbor_overlap(new_neighbors, existing["neighbors"])

    # Predicate/type similarity
    type_match = types_compatible(predict_entity_type(entity_text, predicate), existing["type"])
    p_sim = 1.0 if type_match else 0.0

    # Composite score
    composite_score = alpha * s_sim + beta * e_sim + gamma * n_sim + delta * p_sim
    return composite_score


def resolve_entity(entity_text, role, predicate, other_entity_id=None):
    """
    Resolve an entity by finding the best candidate to MERGE with or deciding to INSERT a new entity.
    """
    global entity_id_counter

    candidates = semantic_blocking(entity_text, entity_store)

    best_candidate = None
    best_score = 0.0

    new_type = predict_entity_type(entity_text, predicate)

    for c in candidates:
        existing = entity_store[c["entity_id"]]
        existing_type = existing["type"]

        # HARD TYPE GATE
        if not types_compatible(new_type, existing_type):
            continue

        score = composite_similarity(
            entity_text,
            existing,
            predicate=predicate,
            other_entity_id=other_entity_id
        )

        if score > best_score and score >= 0.75:
            best_candidate = c["entity_id"]
            best_score = score

    if best_candidate is not None:
        return best_candidate, "MERGE"

    # INSERT
    new_id = f"E{entity_id_counter}"
    entity_id_counter += 1
    entity_store[new_id] = {
        "name": entity_text,
        "type": new_type,
        "neighbors": set()
    }
    return new_id, "INSERT"

def insert_edge(subject_id, predicate, object_id):
    """
    Insert edge between subject and object, and update their neighbors.
    This function is called after deciding to accept the edge (no duplicate).
    """
    entity_store[subject_id]["neighbors"].add(f"{predicate}:{object_id}")
    entity_store[object_id]["neighbors"].add(f"{predicate}:{subject_id}")


In [ ]:
# ----------------------------
# Toy Dataset
# ----------------------------
toy_dataset = [
    {"subject": "Apple", "predicate": "founded by", "object": "Steve Jobs"},
    {"subject": "Apple", "predicate": "harvested in", "object": "California"},
]

# ----------------------------
# Run the pipeline
# ----------------------------
for triple in toy_dataset:
    subject_text = triple["subject"]
    object_text = triple["object"]
    predicate = triple["predicate"]

    print("========================")
    print("Resolving triple:", triple)
    # resolve subject first
    subj_id, subj_action = resolve_entity(
        subject_text,
        role="subject",
        predicate=predicate
    )

    # resolve object (can use subject as neighbor)
    obj_id, obj_action = resolve_entity(
        object_text,
        role="object",
        predicate=predicate,
        other_entity_id=subj_id
    )

    # canonical_pred = PREDICATE_MAP.get(predicate, predicate)
    edge_key = (subj_id, predicate, obj_id)

    if edge_key in existing_edge_keys:
        excluded_edges.append({
            **triple,
            "reason": "duplicate_edge",
            "subject_id": subj_id,
            "object_id": obj_id
        })
        continue

    accepted_edges.append({
        **triple,
        "subject_id": subj_id,
        "object_id": obj_id,
        "subject_action": subj_action,
        "object_action": obj_action
    })

    existing_edge_keys.add(edge_key)

    # update neighbors (MERGE semantics)
    entity_store[subj_id]["neighbors"].add(f"{predicate}:{obj_id}")
    entity_store[obj_id]["neighbors"].add(f"{predicate}:{subj_id}")

# ----------------------------
# Print results
# ----------------------------
print("=== Accepted Edges ===")
for e in accepted_edges:
    print(e)

print("\n=== Excluded Edges ===")
for e in excluded_edges:
    print(e)

print("\n=== Entity Store ===")
for eid, info in entity_store.items():
    print(eid, info['name'], info['neighbors'], info['type'])

## Evaluation

Stress test dataset

In [148]:
stress_test_dataset_with_labels = [

    # =========================
    # 1. Homonym & Lexical Collision
    # =========================
    {"subject": "Apple", "predicate": "released", "object": "iPhone 15", "gold_cluster": "APPLE_TECH"},
    {"subject": "Apple", "predicate": "harvested in", "object": "September", "gold_cluster": "APPLE_FRUIT"},
    {"subject": "Apple", "predicate": "is a type of", "object": "Fruit", "gold_cluster": "APPLE_FRUIT"},

    # # =========================
    # # 2. Abbreviation Ambiguity
    # # =========================
    # {"subject": "US", "predicate": "invaded", "object": "Iraq", "gold_cluster": "UNITED_STATES"},
    # {"subject": "US", "predicate": "won", "object": "US Open", "gold_cluster": "US_TENNIS_PLAYER"},
    # {"subject": "United States", "predicate": "has capital", "object": "Washington DC", "gold_cluster": "UNITED_STATES"},

    # # =========================
    # # 3. Semantic Embedding Trap
    # # =========================
    # {"subject": "Amazon", "predicate": "develops", "object": "Alexa", "gold_cluster": "AMAZON_COMPANY"},
    # {"subject": "Amazon", "predicate": "headquartered in", "object": "Seattle", "gold_cluster": "AMAZON_COMPANY"},
    # {"subject": "Amazon River", "predicate": "flows through", "object": "Brazil", "gold_cluster": "AMAZON_RIVER"},

    # # =========================
    # # 4. Role vs Entity Confusion
    # # =========================
    # {"subject": "President", "predicate": "leads", "object": "United States", "gold_cluster": "US_PRESIDENT_ROLE"},
    # {"subject": "Joe Biden", "predicate": "is", "object": "President", "gold_cluster": "JOE_BIDEN"},
    # {"subject": "Joe Biden", "predicate": "born in", "object": "Scranton", "gold_cluster": "JOE_BIDEN"},

    # # =========================
    # # 5. Predicate Paraphrase
    # # =========================
    # {"subject": "Elon Musk", "predicate": "leads", "object": "SpaceX", "gold_cluster": "ELON_MUSK"},
    # {"subject": "Elon Musk", "predicate": "is CEO of", "object": "SpaceX", "gold_cluster": "ELON_MUSK"},
    # {"subject": "Elon Musk", "predicate": "runs", "object": "SpaceX", "gold_cluster": "ELON_MUSK"},

    # # =========================
    # # 6. Predicate Polarity Conflict
    # # =========================
    # {"subject": "Company X", "predicate": "acquired", "object": "Company Y", "gold_cluster": "COMPANY_X"},
    # {"subject": "Company Y", "predicate": "acquired", "object": "Company X", "gold_cluster": "COMPANY_Y"},

    # # =========================
    # # 7. Structural / Neighborhood Deception
    # # =========================
    # {"subject": "Paris", "predicate": "located in", "object": "France", "gold_cluster": "PARIS_FRANCE"},
    # {"subject": "Paris", "predicate": "has river", "object": "Seine", "gold_cluster": "PARIS_FRANCE"},
    # {"subject": "Paris", "predicate": "located in", "object": "Texas", "gold_cluster": "PARIS_TEXAS"},
    # {"subject": "Paris", "predicate": "has population", "object": "25000", "gold_cluster": "PARIS_TEXAS"},

    # # =========================
    # # 8. Copycat Organization
    # # =========================
    # {"subject": "OpenAI", "predicate": "develops", "object": "ChatGPT", "gold_cluster": "OPENAI"},
    # {"subject": "OpenAI Research", "predicate": "develops", "object": "ChatGPT", "gold_cluster": "OPENAI_RESEARCH"},
    # {"subject": "OpenAI Research", "predicate": "published", "object": "Paper A", "gold_cluster": "OPENAI_RESEARCH"},
    # {"subject": "OpenAI", "predicate": "published", "object": "Paper B", "gold_cluster": "OPENAI"},

    # # =========================
    # # 9. Temporal Drift
    # # =========================
    # {"subject": "Google", "predicate": "CEO", "object": "Larry Page", "gold_cluster": "GOOGLE"},
    # {"subject": "Google", "predicate": "CEO", "object": "Sundar Pichai", "gold_cluster": "GOOGLE"},

    # # =========================
    # # 10. Compound Adversarial Case
    # # =========================
    # {"subject": "Meta", "predicate": "formerly known as", "object": "Facebook", "gold_cluster": "META"},
    # {"subject": "Facebook", "predicate": "owns", "object": "Instagram", "gold_cluster": "META"},
    # {"subject": "Meta AI", "predicate": "develops", "object": "LLaMA", "gold_cluster": "META_AI"},
    # {"subject": "Facebook AI Research", "predicate": "develops", "object": "LLaMA", "gold_cluster": "FAIR"},
]

# for triple in stress_test_dataset_with_labels:
#     subject_text = triple["subject"]
#     object_text = triple["object"]
#     predicate = triple["predicate"]

#     print("========================")
#     print("Resolving triple:", triple)
#     # resolve subject first
#     subj_id, subj_action = resolve_entity(
#         subject_text,
#         role="subject",
#         predicate=predicate
#     )

#     # resolve object (can use subject as neighbor)
#     obj_id, obj_action = resolve_entity(
#         object_text,
#         role="object",
#         predicate=predicate,
#         other_entity_id=subj_id
#     )

#     # canonical_pred = PREDICATE_MAP.get(predicate, predicate)
#     edge_key = (subj_id, predicate, obj_id)

#     if edge_key in existing_edge_keys:
#         excluded_edges.append({
#             **triple,
#             "reason": "duplicate_edge",
#             "subject_id": subj_id,
#             "object_id": obj_id
#         })
#         continue

#     accepted_edges.append({
#         **triple,
#         "subject_id": subj_id,
#         "object_id": obj_id,
#         "subject_action": subj_action,
#         "object_action": obj_action
#     })

#     existing_edge_keys.add(edge_key)

#     # update neighbors (MERGE semantics)
#     entity_store[subj_id]["neighbors"].add(f"{predicate}:{obj_id}")
#     entity_store[obj_id]["neighbors"].add(f"{predicate}:{subj_id}")

# ----------------------------
# Print results
# ----------------------------
# print("=== Accepted Edges ===")
# for e in accepted_edges:
#     print(e)

# print("\n=== Excluded Edges ===")
# for e in excluded_edges:
#     print(e)

# print("\n=== Entity Store ===")
# for eid, info in entity_store.items():
#     print(eid, info['name'], info['neighbors'])


### Evaluation functions

(Note): There is something wrong with clustering. It also associates the objects with the cluster of the subjects.

Fix:
- Add a separate gold cluster for objects.

In [ ]:
from itertools import combinations
from collections import defaultdict

# ----------------------------
# Helper: Build cluster mapping
# ----------------------------
def build_cluster_map(entity_store, toy_dataset):
    """
    Map each resolved entity ID to its gold cluster
    """
    gold_map = {}
    for triple in toy_dataset:
        # subject
        gold_map[triple["subject"]] = triple["gold_cluster"]
        # object
        gold_map[triple["object"]] = triple["gold_cluster"]
    return gold_map

# ----------------------------
# Helper: Generate all pairwise links for evaluation
# ----------------------------
def pairwise_links(items):
    """
    Returns all pairwise combinations (unordered) of items
    """
    return set(frozenset([a, b]) for a, b in combinations(items, 2))

# ----------------------------
# Evaluation function
# ----------------------------
def evaluate_er_pipeline(resolved_entities, toy_dataset):
    """
    resolved_entities: dict mapping entity_text -> resolved_entity_id
    toy_dataset: list of dicts with 'gold_cluster'
    """

    # Build gold clusters
    gold_clusters = defaultdict(set)
    for triple in toy_dataset:
        gold_clusters[triple["gold_cluster"]].add(triple["subject"])
        gold_clusters[triple["gold_cluster"]].add(triple["object"])

    # Build predicted clusters
    pred_clusters = defaultdict(set)
    for entity_text, eid in resolved_entities.items():
        pred_clusters[eid].add(entity_text)

    # Build pairwise links
    gold_links = set()
    for cluster_entities in gold_clusters.values():
        gold_links.update(pairwise_links(cluster_entities))

    pred_links = set()
    for cluster_entities in pred_clusters.values():
        pred_links.update(pairwise_links(cluster_entities))

    # Compute pairwise Precision / Recall / F1
    true_positive = len(gold_links & pred_links)
    false_positive = len(pred_links - gold_links)
    false_negative = len(gold_links - pred_links)

    precision = true_positive / (true_positive + false_positive + 1e-10)
    recall = true_positive / (true_positive + false_negative + 1e-10)
    f1 = 2 * precision * recall / (precision + recall + 1e-10)

    print(f"Evaluation Results:")
    print(f"Pairwise Precision: {precision:.3f}")
    print(f"Pairwise Recall   : {recall:.3f}")
    print(f"Pairwise F1       : {f1:.3f}")
    print()
    
    
    print("Gold Clusters:")
    for cluster_id, entities in gold_clusters.items():
        print(f"{cluster_id}: {entities}")

    print("\nPredicted Clusters:")
    for cluster_id, entities in pred_clusters.items():  
        print(f"{cluster_id}: {entities}")


    # # Optional: log problematic merges
    # problem_merges = []
    # for cluster in pred_clusters.values():
    #     print("Checking predicted cluster:", cluster)
    #     # Check if merged entities come from multiple gold clusters
    #     golds = set(toy_dataset[i]["gold_cluster"] for i, t in enumerate(toy_dataset) if t["subject"] in cluster or t["object"] in cluster)
    #     if len(golds) > 1:
    #         problem_merges.append({
    #             "predicted_cluster": cluster,
    #             "gold_clusters": golds
    #         })
    # if problem_merges:
    #     print("Problematic merges detected:")
    #     for pm in problem_merges:
    #         print(pm)
    # else:
    #     print("No problematic merges detected")

    # Optional: log problematic merges
    problem_merges = []
    for pred_cluster_id, pred_entities in pred_clusters.items():
        for gold_cluster_id, gold_entities in gold_clusters.items():
            if pred_entities & gold_entities and not (pred_entities <= gold_entities or gold_entities <= pred_entities):  # if they overlap but neither is a subset of the other
                problem_merges.append({
                    "predicted_cluster": pred_cluster_id, "predicted_entities": pred_entities,
                    "gold_cluster": gold_cluster_id, "gold_entities": gold_entities
                })

    if problem_merges:
        print("Problematic merges detected:")
        for i, pm in enumerate(problem_merges):
            print(f"Problem {i+1}: {pm['predicted_cluster']} (entities: {pm['predicted_entities']}) overlaps with {pm['gold_cluster']} (entities: {pm['gold_entities']})")
    else:
        print("No problematic merges detected")


### Steps in using evaluation
Step 1. Create resolved entities variable for evaluation

In [150]:
## These are from the entity store after running the pipeline

## Example
# resolved_entities = {
#     "Apple": "E1",
#     "Apple Fruit": "E2",
#     "US": "E3",
#     "United States": "E3",
#     "Joe Biden": "E4",
#     "President": "E5",
# }

# Build resolved_entities mapping from your entity_store
# resolved_entities = {}

# for eid, info in entity_store.items():
#     entity_name = info["name"]
#     resolved_entities[entity_name] = eid

# print(resolved_entities)

resolved_entities = {}

for eid, info in entity_store.items():
    entity_name = info["name"] 
    resolved_entities[entity_name] = eid

# for eid, info in entity_store.items():
#     entity_name = info["name"] + f"_{info['type']}"
#     resolved_entities[entity_name] = eid

print(resolved_entities)

{'Apple': 'E5', 'iPhone 15': 'E2', 'September': 'E4', 'Fruit': 'E6'}


Step 2: Call function

In [151]:
## This is the main function call to evaluate
evaluate_er_pipeline(resolved_entities, stress_test_dataset_with_labels)

Evaluation Results:
Pairwise Precision: 0.000
Pairwise Recall   : 0.000
Pairwise F1       : 0.000

Gold Clusters:
APPLE_TECH: {'Apple', 'iPhone 15'}
APPLE_FRUIT: {'Apple', 'September', 'Fruit'}

Predicted Clusters:
E5: {'Apple'}
E2: {'iPhone 15'}
E4: {'September'}
E6: {'Fruit'}
Problematic merges detected:
{'predicted_cluster': {'Apple'}, 'gold_clusters': {'APPLE_TECH', 'APPLE_FRUIT'}}


Example

In [ ]:
resolved_entities_sample = {

    # =========================
    # Apple (homonym)
    # =========================
    "Apple (tech)": "APPLE_TECH",
    "Apple": "APPLE_TECH",          # when used in tech context
    "iPhone 15": "IPHONE_15",

    "Apple (fruit)": "E_APPLE_FRUIT",
    "Fruit": "E_FRUIT",
    "September": "E_SEPTEMBER",

    # # =========================
    # # United States vs US (ambiguous)
    # # =========================
    # "US (country)": "E_UNITED_STATES",
    # "United States": "E_UNITED_STATES",
    # "Washington DC": "E_WASHINGTON_DC",

    # "US (tennis player)": "E_US_TENNIS_PLAYER",
    # "US Open": "E_US_OPEN",

    # "Iraq": "E_IRAQ",

    # # =========================
    # # Amazon
    # # =========================
    # "Amazon": "E_AMAZON_COMPANY",
    # "Alexa": "E_ALEXA",
    # "Seattle": "E_SEATTLE",

    # "Amazon River": "E_AMAZON_RIVER",
    # "Brazil": "E_BRAZIL",

    # # =========================
    # # Joe Biden / President role
    # # =========================
    # "Joe Biden": "E_JOE_BIDEN",
    # "Scranton": "E_SCRANTON",

    # "President": "E_US_PRESIDENT_ROLE",

    # # =========================
    # # Elon Musk / SpaceX
    # # =========================
    # "Elon Musk": "E_ELON_MUSK",
    # "SpaceX": "E_SPACEX",

    # # =========================
    # # Company X / Company Y
    # # =========================
    # "Company X": "E_COMPANY_X",
    # "Company Y": "E_COMPANY_Y",

    # # =========================
    # # Paris ambiguity
    # # =========================
    # "Paris (France)": "E_PARIS_FRANCE",
    # "France": "E_FRANCE",
    # "Seine": "E_SEINE",

    # "Paris (Texas)": "E_PARIS_TEXAS",
    # "Texas": "E_TEXAS",
    # "25000": "E_POPULATION_25000",

    # # =========================
    # # OpenAI vs OpenAI Research
    # # =========================
    # "OpenAI": "E_OPENAI",
    # "ChatGPT": "E_CHATGPT",
    # "Paper B": "E_PAPER_B",

    # "OpenAI Research": "E_OPENAI_RESEARCH",
    # "Paper A": "E_PAPER_A",

    # # =========================
    # # Google temporal drift
    # # =========================
    # "Google": "E_GOOGLE",
    # "Larry Page": "E_LARRY_PAGE",
    # "Sundar Pichai": "E_SUNDAR_PICHAI",

    # # =========================
    # # Meta / Facebook / AI orgs
    # # =========================
    # "Meta": "E_META",
    # "Facebook": "E_META",
    # "Instagram": "E_INSTAGRAM",

    # "Meta AI": "E_META_AI",
    # "Facebook AI Research": "E_FAIR",
    # "LLaMA": "E_LLAMA",
}

## This is the main function call to evaluate
evaluate_er_pipeline(resolved_entities_sample, stress_test_dataset_with_labels)


Sample Correct predictions

In [171]:
stress_test_dataset_with_labels = [

    # =========================
    # 1. Homonym & Lexical Collision
    # =========================
    {"subject": "Apple Inc", "predicate": "released", "object": "iPhone 15", "gold_cluster": "APPLE_TECH"},
    {"subject": "Apple", "predicate": "harvested in", "object": "September", "gold_cluster": "APPLE_FRUIT"},
    {"subject": "Apple", "predicate": "is a type of", "object": "Fruit", "gold_cluster": "APPLE_FRUIT"},]

resolved_entities_sample = {
    "Apple Inc": "APPLE_TECH",  # incorrectly merged into tech cluster
    "iPhone 15": "APPLE_TECH",
    "Apple": "APPLE_FRUIT",
    "September": "APPLE_FRUIT",    
    "Fruit": "APPLE_FRUIT",
}

evaluate_er_pipeline(resolved_entities_sample, stress_test_dataset_with_labels)

Evaluation Results:
Pairwise Precision: 1.000
Pairwise Recall   : 1.000
Pairwise F1       : 1.000

Gold Clusters:
APPLE_TECH: {'Apple Inc', 'iPhone 15'}
APPLE_FRUIT: {'Apple', 'September', 'Fruit'}

Predicted Clusters:
APPLE_TECH: {'Apple Inc', 'iPhone 15'}
APPLE_FRUIT: {'Apple', 'September', 'Fruit'}
No problematic merges detected


In [181]:
stress_test_dataset_with_labels = [

    # =========================
    # 1. Homonym & Lexical Collision
    # =========================
    {"subject": "Apple Inc", "predicate": "released", "object": "iPhone 15", "gold_cluster": "APPLE_TECH"},
    {"subject": "Apple", "predicate": "harvested in", "object": "September", "gold_cluster": "APPLE_FRUIT"},
    {"subject": "Apple", "predicate": "is a type of", "object": "Fruit", "gold_cluster": "APPLE_FRUIT"},]

resolved_entities_sample = {
    "Apple company": "APPLE_TECH",  # incorrectly merged into tech cluster
    "iPhone 15": "APPLE_TECH",
    "Apple fruit": "APPLE_FRUIT",
    "September": "APPLE_FRUIT",    
    "Fruit": "APPLE_FRUIT",
}

evaluate_er_pipeline(resolved_entities_sample, stress_test_dataset_with_labels)

Evaluation Results:
Pairwise Precision: 0.250
Pairwise Recall   : 0.250
Pairwise F1       : 0.250

Gold Clusters:
APPLE_TECH: {'Apple Inc', 'iPhone 15'}
APPLE_FRUIT: {'Apple', 'September', 'Fruit'}

Predicted Clusters:
APPLE_TECH: {'Apple company', 'iPhone 15'}
APPLE_FRUIT: {'September', 'Fruit', 'Apple fruit'}
Problematic merges detected:
Problem 1: APPLE_TECH (entities: {'Apple company', 'iPhone 15'}) overlaps with APPLE_TECH (entities: {'Apple Inc', 'iPhone 15'})
Problem 2: APPLE_FRUIT (entities: {'September', 'Fruit', 'Apple fruit'}) overlaps with APPLE_FRUIT (entities: {'Apple', 'September', 'Fruit'})


In [180]:
stress_test_dataset_with_labels = [

    # =========================
    # 1. Homonym & Lexical Collision
    # =========================
    {"subject": "Apple Inc", "predicate": "released", "object": "iPhone 15", "gold_cluster": "APPLE_TECH"},
    {"subject": "Apple", "predicate": "harvested in", "object": "September", "gold_cluster": "APPLE_FRUIT"},
    {"subject": "Apple", "predicate": "is a type of", "object": "Fruit", "gold_cluster": "APPLE_FRUIT"},]

resolved_entities_sample = {
    "Apple company": "APPLE_TECH",  # incorrectly merged into tech cluster
    "iPhone 15": "APPLE_TECH",
    "Apple fruit": "APPLE_TECH",
    "September": "APPLE_TECH",    
    "Fruit": "APPLE_TECH",
}

evaluate_er_pipeline(resolved_entities_sample, stress_test_dataset_with_labels)

Evaluation Results:
Pairwise Precision: 0.100
Pairwise Recall   : 0.250
Pairwise F1       : 0.143

Gold Clusters:
APPLE_TECH: {'Apple Inc', 'iPhone 15'}
APPLE_FRUIT: {'Apple', 'September', 'Fruit'}

Predicted Clusters:
APPLE_TECH: {'Apple company', 'September', 'Apple fruit', 'iPhone 15', 'Fruit'}
Problematic merges detected:
Problem 1: APPLE_TECH (entities: {'Apple company', 'September', 'Apple fruit', 'iPhone 15', 'Fruit'}) overlaps with APPLE_TECH (entities: {'Apple Inc', 'iPhone 15'})
Problem 2: APPLE_TECH (entities: {'Apple company', 'September', 'Apple fruit', 'iPhone 15', 'Fruit'}) overlaps with APPLE_FRUIT (entities: {'Apple', 'September', 'Fruit'})
